In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import StringType
from pyspark.sql.functions import when, col


In [0]:
df_bronze = spark.table("workspace.default.bronze_diabetic")

print(f"Linhas: {df_bronze.count()}")
df_bronze.printSchema()

Linhas: 101766
root
 |-- encounter_id: integer (nullable = true)
 |-- patient_nbr: integer (nullable = true)
 |-- race: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- admission_type_id: integer (nullable = true)
 |-- discharge_disposition_id: integer (nullable = true)
 |-- admission_source_id: integer (nullable = true)
 |-- time_in_hospital: integer (nullable = true)
 |-- payer_code: string (nullable = true)
 |-- medical_specialty: string (nullable = true)
 |-- num_lab_procedures: integer (nullable = true)
 |-- num_procedures: integer (nullable = true)
 |-- num_medications: integer (nullable = true)
 |-- number_outpatient: integer (nullable = true)
 |-- number_emergency: integer (nullable = true)
 |-- number_inpatient: integer (nullable = true)
 |-- diag_1: string (nullable = true)
 |-- diag_2: string (nullable = true)
 |-- diag_3: string (nullable = true)
 |-- number_diagnoses: integer (nulla

In [0]:
total = df_bronze.count()

resumo = []
for coluna in df_bronze.columns:
    nulos = df_bronze.filter(col(coluna).isNull()).count()
    
    # Just verify "?" in text columns
    if isinstance(df_bronze.schema[coluna].dataType, StringType):
        interrogacao = df_bronze.filter(col(coluna) == "?").count()
    else:
        interrogacao = 0
    
    pct = round((nulos + interrogacao) / total * 100, 2)
    resumo.append((coluna, nulos, interrogacao, pct))

resumo_df = spark.createDataFrame(resumo, ["Column", "nulls", "interrogations", "pct_of_problems"])
resumo_df.filter(col("pct_of_problems") > 0).orderBy(col("pct_of_problems").desc()).display()

Column,nulls,interrogations,pct_of_problems
weight,0,98569,96.86
medical_specialty,0,49949,49.08
payer_code,0,40256,39.56
race,0,2273,2.23
diag_3,0,1423,1.4
diag_2,0,358,0.35
diag_1,0,21,0.02


In [0]:
df_silver = df_bronze

# Remove columns with unseful data
df_silver = df_silver.drop("weight", "payer_code")

# change "?" for "Unknown" in text columns
colunas_unknown = ["race", "medical_specialty", "diag_1", "diag_2", "diag_3"]
for coluna in colunas_unknown:
    df_silver = df_silver.replace("?", "Unknown", subset=[coluna])

# Remove rows where gender is invalid
df_silver = df_silver.filter(col("gender") != "Unknown/Invalid")

print(f"lines after cleaning: {df_silver.count()}")

lines after cleaning: 101763


In [0]:
df_silver = df_silver.withColumn(
    "readmitted_flag",
    when(col("readmitted") == "<30", 1).otherwise(0)
)

df_silver.groupBy("readmitted", "readmitted_flag").count().orderBy("count").display()

readmitted,readmitted_flag,count
<30,1,11357
>30,0,35545
NO,0,54861


In [0]:
# Validation 1: no nulls in encounter_id
assert df_silver.filter(col("encounter_id").isNull()).count() == 0, "encounter_id has null values!"

# Validation 2: time_in_hospital always positive
assert df_silver.filter(col("time_in_hospital") <= 0).count() == 0, "time_in_hospital has invalid values!"

# Validation 3: readmitted_flag only contains 0 or 1
invalid_flags = df_silver.filter(~col("readmitted_flag").isin([0, 1])).count()
assert invalid_flags == 0, "readmitted_flag has unexpected values!"

# Validation 4: we didn't lose more than 1% of rows
assert df_silver.count() >= 101766 * 0.99, "Too many rows lost during cleaning!"

print("All validations passed!")

All validations passed!


In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.default.silver_diabetic")
)

print("Silver saved successfully!")

Silver saved successfully!
